# Algorithm Comparison

This notebook compares the performance of Logistic Regression, Random Forest, XGBoost, and CatBoost on the heart disease dataset.

In [6]:
from pathlib import Path
import pickle

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [7]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for path in [current, *current.parents]:
        # Use the data folder to identify the main project directory.
        if (path / 'data').exists():
            return path

    raise FileNotFoundError('Project root not found.')


# Define all project paths used in this notebook.
BASE_DIR = find_project_root(Path.cwd())
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'heart_disease_processed.csv'
MODEL_DIR = BASE_DIR / 'models'

BASE_DIR, DATA_PATH, MODEL_DIR

(WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/processed/heart_disease_processed.csv'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/models'))

In [8]:
# Load the processed dataset and keep a working copy for evaluation.
data = pd.read_csv(DATA_PATH)
df = data.copy()
df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40.0,1,0,0.0,0,0,...,1,0,5.0,18.0,15.0,1,0,9.0,4.0,3.0
1,0,0,0,0,25.0,1,0,0.0,1,0,...,0,1,3.0,0.0,0.0,0,0,7.0,6.0,1.0
2,0,1,1,1,28.0,0,0,0.0,0,1,...,1,1,5.0,30.0,30.0,1,0,9.0,4.0,8.0
3,0,1,0,1,27.0,0,0,0.0,1,1,...,1,0,2.0,0.0,0.0,0,0,11.0,3.0,6.0
4,0,1,1,1,24.0,0,0,0.0,1,1,...,1,0,2.0,3.0,0.0,0,0,11.0,5.0,4.0


In [9]:
# Separate features and target before creating the test set.
target_column = 'HeartDiseaseorAttack'

X = df.drop(columns=[target_column])
y = df[target_column]

print(f'Feature shape: {X.shape}')
print(f'Target shape: {y.shape}')

Feature shape: (229781, 21)
Target shape: (229781,)


In [10]:
# Split the dataset exactly as the training notebooks do.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_test shape: {X_test.shape}')

X_test shape: (45957, 21)


In [11]:
# Load the saved models and other required artifacts.
logistic_model_path = MODEL_DIR / 'logistic_regression_model.pkl'
logistic_scaler_path = MODEL_DIR / 'logistic_regression_scaler.pkl'

random_forest_model_path = MODEL_DIR / 'random_forest_model.pkl'
xgboost_model_path = MODEL_DIR / 'xgboost_model.pkl'
catboost_model_path = MODEL_DIR / 'catboost_model.pkl'

with open(logistic_model_path, 'rb') as file:
    logistic_model = pickle.load(file)

with open(logistic_scaler_path, 'rb') as file:
    logistic_scaler = pickle.load(file)

with open(random_forest_model_path, 'rb') as file:
    random_forest_model = pickle.load(file)

with open(xgboost_model_path, 'rb') as file:
    xgboost_model = pickle.load(file)

with open(catboost_model_path, 'rb') as file:
    catboost_model = pickle.load(file)

In [12]:
# Scale the test set only for Logistic Regression because that model was trained on scaled data.
X_test_logistic = logistic_scaler.transform(X_test)

In [13]:
def evaluate_model(model_name, model, X_eval, y_true):
    # This helper computes the main evaluation metrics for one model.
    y_pred = model.predict(X_eval)
    y_pred_proba = model.predict_proba(X_eval)[:, 1]

    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_pred_proba)
    }

In [14]:
# Evaluate all four algorithms on the same test set.
results = []

results.append(evaluate_model('Logistic Regression', logistic_model, X_test_logistic, y_test))
results.append(evaluate_model('Random Forest', random_forest_model, X_test, y_test))
results.append(evaluate_model('XGBoost', xgboost_model, X_test, y_test))
results.append(evaluate_model('CatBoost', catboost_model, X_test, y_test))

In [15]:
# Convert the results into a clean comparison table.
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,CatBoost,0.730248,0.250261,0.808560,0.382220,0.845255
1,Logistic Regression,0.739104,0.252307,0.778199,0.381065,0.835465
2,XGBoost,0.737015,0.251034,0.780519,0.379887,0.834951
3,Random Forest,0.891507,0.396596,0.098250,0.157486,0.804384


In [16]:
# Highlight the best model based on ROC-AUC.
best_model_row = comparison_df.iloc[0]
print('Best Model Based on ROC-AUC:')
print(best_model_row)

Best Model Based on ROC-AUC:
Model        CatBoost
Accuracy     0.730248
Precision    0.250261
Recall        0.80856
F1 Score      0.38222
ROC-AUC      0.845255
Name: 0, dtype: object
